# 116. PDF 미성형 원인 기반 비라벨 센서 관계 보조학습
PDF11쪽(인쇄10쪽), 그림3의 공급/충전압/고화/공기저항 가설에 따라 압력·충진시간·쿠션·계량시간의 예상값을 학습. 비라벨을 정상이라고 간주하지 않고 수치센서 예측만 수행. 이후 라벨 정상/미성형 분류.
외부/내부 평가구간의 라벨 관측 시간범위±30분을 비라벨에서 제외. 모든 원래라벨 Timestamp도 제거. 같은설비RG3, 최대4000개 고정표본. 다른 시기의 미래 비라벨은 사용할 수 있어 온라인 미래 예측 검증은 아님. 공정관계는 인과식 아님.
분류기는 각2~4개 잔차 RF/ExtraTrees 2seed 평균. 내부 조합/임계값 선택. 새로운 독립자료가 아닌 반복개발 구간검증이다. 가스모델 변경하지 않음.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
allshots=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=allshots[allshots['product'].eq('RG3')&(allshots.PassOrFail.eq(0)|allshots.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);g=d.comparison_block.to_numpy();blocks=np.unique(g)
assert len(d)==574 and y.sum()==8
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
u=u[u.EQUIP_CD.isin(d.EQUIP_CD.unique())].sort_values('TimeStamp').drop_duplicates(['EQUIP_CD','TimeStamp'])
# All label keys, including excluded gas and other types, are removed.
u=u[~u.TimeStamp.isin(allshots.TimeStamp)].reset_index(drop=True)
relations={
 'pressure':('Max_Injection_Pressure',['Max_Switch_Over_Pressure','Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']),
 'fill':('Filling_Time',['Max_Injection_Speed','Max_Injection_Pressure','Barrel_Temperature_5','Mold_Temperature_3']),
 'cushion':('Cushion_Position',['Plasticizing_Position','Average_Back_Pressure','Max_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5']),
 'recovery':('Plasticizing_Time',['Plasticizing_Position','Average_Back_Pressure','Max_Screw_RPM','Barrel_Temperature_2','Hopper_Temperature'])}
# Numerical relationships, not identified physical causal equations.
features_cache={};model_cache={};audit=[]
def features(ref,q):
 key=(tuple(ref),tuple(q))
 if key in features_cache:return features_cache[key]
 excluded=set(blocks)-set(g[ref])
 usable=np.ones(len(u),bool)
 for b in excluded:
  z=d[g==b];lo=z.TimeStamp.min()-pd.Timedelta(minutes=30);hi=z.TimeStamp.max()+pd.Timedelta(minutes=30)
  usable &= ~u.TimeStamp.between(lo,hi).to_numpy()
 uu=u[usable]
 # Fixed random sample; eligibility independent of unknown quality.
 if len(uu)>4000:uu=uu.sample(4000,random_state=42)
 assert not uu.TimeStamp.isin(d.TimeStamp).any()
 assert len(uu)>100
 raw=[]
 for name,(target,parents) in relations.items():
  valid=uu[target].notna();ut=uu.loc[valid]
  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(ut[parents])
  m=ExtraTreesRegressor(n_estimators=40,max_depth=10,min_samples_leaf=8,random_state=42,n_jobs=2)
  m.fit(imp.transform(ut[parents]),ut[target])
  err=d[target].to_numpy()-m.predict(imp.transform(d[parents]))
  # Scale/center with ref labeled normals, never q labels/statistics.
  normal=ref[y[ref]==0];center=np.median(err[normal]);scale=max(np.std(err[normal]),float(d[target].iloc[normal].std())*.1,1e-6)
  raw.append((err-center)/scale)
 f=np.column_stack(raw)
 features_cache[key]=f
 audit.append(dict(train=len(ref),test=len(q),unlabeled_used=len(uu),excluded_blocks=';'.join(sorted(excluded)),label_timestamp_overlap=0,ref_shortshot=int(y[ref].sum())))
 return f
views={'all_residual4':[0,1,2,3],'fill_pressure2':[0,1],'supply_recovery2':[2,3]}
names=list(views)
def predict(ref,q):
 assert set(g[ref]).isdisjoint(g[q])
 key=(tuple(ref),tuple(q))
 if key in model_cache:return model_cache[key]
 f=features(ref,q);out=[]
 for name,cols in views.items():
  if len(np.unique(y[ref]))<2:out.append(np.zeros(len(q)));continue
  tr=f[ref][:,cols];te=f[q][:,cols];scores=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
    scores.append(m.fit(tr,y[ref]).predict_proba(te)[:,1])
  out.append(np.mean(scores,axis=0))
 result=np.column_stack(out);model_cache[key]=result;return result
weights=[(n,np.eye(3)[j]) for j,n in enumerate(names)]+[('all_experts_mean',np.ones(3)/3)]
def threshold(yy,s):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
 k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(g==b);ref=np.flatnonzero(g!=b);outer=predict(ref,q)
 inner=np.full((len(ref),3),np.nan)
 for ib in np.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for name,w in weights:
  cal=inner@w;score=outer@w;cut,fi=threshold(y[ref],cal);candidates.append((fi,name,w,cut))
  for rule,c in [('inner_F1',cut),('normal99',float(np.quantile(cal[y[ref]==0],.99,method='higher')))]:
   pr=score>=c
   for j,i in enumerate(q):records.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=c,prediction=int(pr[j])))
 fi,name,w,c=max(candidates,key=lambda z:(z[0],-np.count_nonzero(z[2]),z[1]));score=outer@w;pr=score>=c
 choices.append(dict(block=b,selected=name,inner_F1=fi,threshold=c))
 for j,i in enumerate(q):records.append(dict(model='primary_inner_selected',rule='inner_F1',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=c,prediction=int(pr[j])))
 print('Completed unlabeled teacher',b,name,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,rule),z in p.groupby(['model','rule']):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,rule=rule,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(audit),'training_audit'),(pd.DataFrame(choices),'choices')]:
 obj.to_csv(t/('rg3_shortshot_unlabeled_teacher_'+name+'.csv'),index=False,encoding='utf-8-sig')
# Each row's residuals from its own held-out-block teacher only.
caseframes=[]
for b in blocks:
 ref=np.flatnonzero(g!=b);q=np.flatnonzero(g==b);f=features(ref,q)
 caseframes.append(pd.DataFrame(f[q],columns=list(relations)).assign(TimeStamp=d.TimeStamp.iloc[q].to_numpy(),block=b,actual=y[q]))
res=pd.concat(caseframes)
res.to_csv(t/'rg3_shortshot_unlabeled_teacher_residuals.csv',index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False),flush=True)
print('ALL8 residuals',res[res.actual.eq(1)].to_string(index=False),flush=True)


Completed unlabeled teacher 20201021_001626_RG3 fill_pressure2


Completed unlabeled teacher 20201022_002405_RG3 fill_pressure2


Completed unlabeled teacher 20201022_044026_RG3 fill_pressure2


Completed unlabeled teacher 20201023_010128_RG3 fill_pressure2


Completed unlabeled teacher 20201023_044148_RG3 fill_pressure2


                 model     rule  TP  FP  FN  TN       F1       AP
        fill_pressure2 inner_F1   7 326   1 240 0.041056 0.018413
primary_inner_selected inner_F1   7 326   1 240 0.041056 0.018413
      all_experts_mean inner_F1   5 339   3 227 0.028409 0.012347
         all_residual4 inner_F1   3 240   5 326 0.023904 0.012122
      supply_recovery2 inner_F1   5 502   3  64 0.019417 0.011509
         all_residual4 normal99   0  10   8 556 0.000000 0.012122
      all_experts_mean normal99   0  12   8 554 0.000000 0.012347
        fill_pressure2 normal99   0  19   8 547 0.000000 0.018413
      supply_recovery2 normal99   0   9   8 557 0.000000 0.011509


ALL8 residuals  pressure      fill   cushion  recovery           TimeStamp               block  actual
-0.451743 -1.584826  1.815953  0.972159 2020-10-22 01:09:23 20201022_002405_RG3       1
 0.343416  0.102751  0.715929  1.450725 2020-10-22 04:48:39 20201022_044026_RG3       1
-0.310829 -0.927972  1.380841  1.068009 2020-10-22 05:42:13 20201022_044026_RG3       1
 0.097034 -0.030229 -0.388369  1.603433 2020-10-22 06:12:03 20201022_044026_RG3       1
-0.556533 -1.305219  0.664034  0.751517 2020-10-22 06:58:22 20201022_044026_RG3       1
 1.507095 -0.221689 -1.491650 -0.246822 2020-10-23 02:25:54 20201023_010128_RG3       1
 0.723414  0.752052 -1.267880 -0.930986 2020-10-23 05:47:40 20201023_044148_RG3       1
-1.425358  1.633404 -0.582372 -0.541848 2020-10-23 05:54:52 20201023_044148_RG3       1
